# Chapter 7 · Managing context and cost

Count tokens and estimate the cost, measure how long the model takes to read and to write, compare five ways of remembering a long conversation and try two caches: the prefix cache and the answer cache.

**Before you start, turn on the GPU:** menu *Runtime* → *Change runtime type* → *T4 GPU* → *Save*. The notebook loads the language model and the embedding model. Your times will differ from the book's, but the proportions should be similar.

The full explanation is in the PDF chapter. This notebook is the lab.

📘 The book on Amazon: [paperback](https://www.amazon.com/dp/B0HMX9N1KQ) · [Kindle](https://www.amazon.com/dp/B0HMX44VJ4)

## 1. Load

In [ ]:
"""Chapter 7: Managing context and cost. Open model, on a GPU or a CPU."""
import copy
import json
import re
import time
import unicodedata
import urllib.request
from pathlib import Path

import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

GPU_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # with Colab's free T4 GPU
CPU_MODEL = "Qwen/Qwen3-0.6B"             # without a GPU: smaller and faster
MODEL = None                              # None: choose according to the hardware
EMBEDDING_MODEL = "intfloat/multilingual-e5-base"
DATA = Path("data")
REPOSITORY = "https://raw.githubusercontent.com/HernanDiaz/llm-engineering/main/en/"



def load(name=None):
    """The language model and the embedding model, on the GPU if there is one."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if name is None:
        name = GPU_MODEL if device == "cuda" else CPU_MODEL
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32)
    model.to(device).eval()
    embeddings = SentenceTransformer(EMBEDDING_MODEL, device=device)
    print(f"Model {name} on {device}")
    return tok, model, embeddings


def download(name):
    """A file from the data folder; in Colab, it is downloaded from the repository."""
    path = DATA / name
    if not path.exists():
        DATA.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORY + "data/" + name, path)
    return path.read_text(encoding="utf-8")


def synchronize():
    """Wait for the GPU to finish, so that the time is measured correctly."""
    if torch.cuda.is_available():
        torch.cuda.synchronize()

In [ ]:
tok, model, embeddings = load(MODEL)

## 2. Tokens, time and money

In [ ]:
def tokens(text):
    """Number of tokens in a text."""
    return len(tok(text, add_special_tokens=False)["input_ids"])


def conversation_tokens(conversation):
    """Tokens of the whole conversation, with the chat template."""
    text = tok.apply_chat_template(conversation, tokenize=False,
                                   add_generation_prompt=True,
                                   enable_thinking=False)
    return tokens(text)


# Example prices of a commercial API, in dollars per million tokens.
INPUT_PRICE, OUTPUT_PRICE = 0.40, 1.60


def cost(input_tokens, output_tokens):
    """Dollars a request would cost with the example prices."""
    return (input_tokens * INPUT_PRICE + output_tokens * OUTPUT_PRICE) / 1e6

In [ ]:
for sentence in ["How much time do I have to return a bike if I am not happy with it?",
                 "¿Cuánto tiempo tengo para devolver una bicicleta si no me convence?"]:
    print(len(sentence.split()), "words,", tokens(sentence), "tokens:", sentence)
terms = download("terms_and_conditions.md")
print("Terms and conditions:", len(terms.split()), "words,", tokens(terms), "tokens")
print(f"Cost of reading 6199 tokens and writing 386: ${cost(6199, 386):.4f}")

In [ ]:
def generate(conversation, max_tokens=150, cache=None):
    """Answer a conversation. Return the answer, the input and output tokens and
    the seconds. With cache, reuse the computation of a prefix."""
    text = tok.apply_chat_template(conversation, tokenize=False,
                                   add_generation_prompt=True,
                                   enable_thinking=False)
    inputs = tok(text, return_tensors="pt").input_ids.to(model.device)
    extra = {"past_key_values": copy.deepcopy(cache)} if cache is not None else {}
    synchronize()
    start = time.time()
    with torch.no_grad():
        output = model.generate(inputs, max_new_tokens=max_tokens, do_sample=False,
                                temperature=None, top_p=None, top_k=None,
                                pad_token_id=tok.eos_token_id, **extra)
    synchronize()
    new = output[0, inputs.shape[1]:]
    answer = tok.decode(new, skip_special_tokens=True)
    answer = re.sub(r"<think>.*?</think>", "", answer, flags=re.S).strip()
    return answer, inputs.shape[1], len(new), time.time() - start


def time_by_length(lengths=(250, 500, 1000, 2000, 4000), output=50):
    """Seconds to read prompts of different lengths (with a single output token)
    and to also generate 50 tokens. Each measurement is repeated three times
    and the median is taken, to remove noise."""
    filler = download("terms_and_conditions.md") * 6
    rows = []
    for n in lengths:
        ids = tok(filler, add_special_tokens=False)["input_ids"][:n]
        text = tok.decode(ids) + "\n\nSummarize the text."
        messages = [{"role": "user", "content": text}]
        generate(messages, max_tokens=1)                      # warm up
        read = sorted(generate(messages, max_tokens=1)[3] for _ in range(3))[1]
        inputs = generate(messages, max_tokens=1)[1]
        measures = sorted((generate(messages, max_tokens=output)[2:] for _ in range(3)),
                          key=lambda m: m[1])
        generated, total = measures[1]
        rows.append({"tokens": inputs, "read": read, "read_and_generate": total,
                     "generated": generated})
        print(f"{inputs:5d} tokens: read {read:.2f} s; read and generate "
              f"{generated} tokens {total:.2f} s")
    return rows

In [ ]:
times = time_by_length()

## 3. A long conversation and five memories

In [ ]:
POLICIES = (
    "Shipping: to the contiguous US in 2-3 business days, free on orders over "
    "$50. To Hawaii, in 5-7 business days, for $14.95. "
    "Returns: 30 days from delivery. Warranty: 2 years. Repair shop: basic "
    "tune-up for $35, the first one free for bikes bought here. "
    "Interest-free installments on purchases over $300. Store hours: Monday to "
    "Friday from 10 a.m. to 8 p.m. and Saturdays from 10 a.m. to 2 p.m.")


SYSTEM = ("You are the customer service assistant of North Cycles, a bicycle "
          "store. Answer in English, in one or two sentences. You can use "
          "everything the customer has told you in the conversation and these "
          "store details:\n" + POLICIES)


# Each customer turn and, if something is checked, the texts the answer must
# contain (one is enough).
CONVERSATION = [
    ("Hi, I'm Martha Ingram. Last month I bought an Aero 3 bike, "
     "order NC-11020.", None),
    ("How long do I have to return it if I'm not happy with it?",
     ["30 days", "30-day"]),
    ("OK. Something else: do you do tune-ups in your repair shop?", None),
    ("And how much does it cost?", ["$35", "35 dollars", "free"]),
    ("By the way, my brother lives in Honolulu. Could you send him a helmet?",
     None),
    ("How long would it take to get to him?", ["5-7", "5 to 7"]),
    ("Sorry, I've lost the confirmation email. What was my order "
     "number?", ["NC-11020"]),
    ("Can you remind me which bike I told you I had? I want to buy accessories "
     "for it.", ["Aero 3"]),
    ("Can I pay for the accessories in installments?", ["300"]),
    ("Are you open on Saturdays?", ["10 a.m.", "10 am", "10am", "2 p.m.",
                                    "2 pm", "2pm", "10:00"]),
    ("One last thing: whose name is the order under?", ["Martha"]),
    ("Thanks, that's all.", None),
]


SUMMARIZE = ("Summarize in three or four sentences the important facts of this "
             "customer service conversation: the customer's name, their orders "
             "and products, and what they have asked. Copy the names and the "
             "numbers exactly.")


EXTRACT = ("Extract these details from the customer's messages and answer only "
           'with a JSON object: {"name": ..., "order": ..., "product": ...}. '
           "Use null for any that do not appear.")

In [ ]:
def memory_none(history, state):
    """Nothing: only the current message."""
    return []


def memory_full(history, state):
    """All the previous messages."""
    return history


def memory_window(history, state, turns=2):
    """Only the last turns (each turn is two messages)."""
    return history[-2 * turns:]


def memory_summary(history, state, turns=2):
    """The last turns and a summary of the earlier ones, which is updated each
    time a turn leaves the window."""
    old = history[:-2 * turns]
    if len(old) > state.get("summarized", 0):
        text = "\n".join(f"{m['role']}: {m['content']}" for m in old)
        previous = state.get("summary", "")
        state["summary"] = generate(
            [{"role": "system", "content": SUMMARIZE},
             {"role": "user", "content": f"{previous}\n{text}".strip()}],
            max_tokens=120)[0]
        state["summarized"] = len(old)
        state["extra_calls"] = state.get("extra_calls", 0) + 1
    if not old:
        return history
    return ([{"role": "system", "content": "Summary of the earlier conversation: "
              + state["summary"]}] + history[-2 * turns:])


def memory_record(history, state, turns=2):
    """The last turns and a small customer record with a few key facts, extracted
    by the model in JSON and accumulated. A fact is only saved if it appears
    verbatim in the message, if the order has the NC-12345 format and if it had
    not been saved before: that way, no made-up values get in and the good ones
    are not overwritten."""
    record = state.setdefault("record", {})
    if history:
        last = history[-2]["content"]              # last customer message
        try:
            new = json.loads(re.search(r"\{.*\}", generate(
                [{"role": "system", "content": EXTRACT},
                 {"role": "user", "content": last}], max_tokens=60)[0], re.S)[0])
            for key, value in new.items():
                valid = (isinstance(value, str) and value.lower() in last.lower()
                         and (key != "order" or re.fullmatch(r"NC-\d{5}", value)))
                if valid and key not in record:
                    record[key] = value
        except (TypeError, ValueError):
            pass
        state["extra_calls"] = state.get("extra_calls", 0) + 1
    card = json.dumps(record, ensure_ascii=False)
    return ([{"role": "system", "content": "Customer record: " + card}]
            + history[-2 * turns:])


def converse(memory):
    """Go through the conversation with a memory and check the answers."""
    history, state, rows = [], {}, []
    for message, keys in CONVERSATION:
        context = memory(history, state)
        conversation = ([{"role": "system", "content": SYSTEM}] + context
                        + [{"role": "user", "content": message}])
        answer, inputs, outputs, seconds = generate(conversation)
        history += [{"role": "user", "content": message},
                    {"role": "assistant", "content": answer}]
        ok = None if keys is None else any(k.lower() in answer.lower()
                                           for k in keys)
        rows.append({"customer": message, "answer": answer, "ok": ok,
                     "input_tokens": inputs, "seconds": seconds})
    return rows, state


MEMORIES = {"No memory": memory_none,
            "Full history": memory_full,
            "2-turn window": memory_window,
            "Summary + window": memory_summary,
            "Customer record + window": memory_record}

In [ ]:
memories = {}
for name, memory in MEMORIES.items():
    rows, state = converse(memory)
    memories[name] = (rows, state)
    ok = sum(r["ok"] for r in rows if r["ok"] is not None)
    inputs = sum(r["input_tokens"] for r in rows)
    print(f"{name:24s} {ok}/8 correct, {inputs} input tokens, "
          f"{state.get('extra_calls', 0)} extra calls")

Read the whole conversation with one of the memories, with a mark on the answers that fail:

In [ ]:
rows, state = memories["Customer record + window"]
for r in rows:
    mark = "✗ " if r["ok"] is False else "  "
    print(f"{mark}Customer: {r['customer']}\n   Assistant: {r['answer']}")
print("Final state of the memory:", state)

## 4. Two caches

In [ ]:
def with_prefix_cache(system, questions, max_tokens=40):
    """Answer several questions with the same system message, with and without
    reusing the computation of that prefix."""
    prefix = tok.apply_chat_template([{"role": "system", "content": system}],
                                     tokenize=False)
    ids = tok(prefix, return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        cache = model(ids, use_cache=True).past_key_values
    results = {}
    for name, c in (("without cache", None), ("with cache", cache)):
        generate([{"role": "user", "content": "Hello"}], max_tokens=1)   # warm up
        answers, total = [], 0
        for q in questions:
            a, _, _, s = generate([{"role": "system", "content": system},
                                   {"role": "user", "content": q}], max_tokens, c)
            answers.append(a)
            total += s
        results[name] = {"seconds": total, "answers": answers}
        print(f"{name}: {total:.2f} s for {len(questions)} questions")
    return ids.shape[1], results

In [ ]:
catalog = json.loads(download("catalog.json"))
system = ("You are the North Cycles assistant. Answer in one sentence using "
          "the details of this catalog:\n" + "\n".join(
              f"{p['name']} ({p['ref']}), ${p['price']}. {p['description']}"
              for p in catalog))
questions = ["How much is the Aero 3?", "What is the range of the E-Folding?",
             "Do you have helmets for kids?", "How much does the Folding 20 weigh?",
             "Which lock is the most secure?", "What age is the Junior 24 for?"]
length, prefix = with_prefix_cache(system, questions)
print("Prefix tokens:", length)
print("Same answers?", prefix["without cache"]["answers"] == prefix["with cache"]["answers"])

In [ ]:
def normalize(text):
    """Lowercase, without accents or punctuation, to compare questions."""
    text = "".join(c for c in unicodedata.normalize("NFD", text.lower())
                   if unicodedata.category(c) != "Mn")
    return re.sub(r"[^a-z0-9 ]", "", text).strip()


# Questions that arrive one after another, with the intent they correspond to
# (to know whether a stored answer is valid or not).
ARRIVALS = [
    ("How long does shipping to Hawaii take?", "hawaii"),
    ("How much is a tune-up at the repair shop?", "tune_up"),
    ("How long does shipping to Hawaii take?", "hawaii"),
    ("Can I pay in installments?", "installments"),
    ("how long does shipping to hawaii take", "hawaii"),
    ("How many days do I have to return a product?", "return"),
    ("Is it possible to pay in installments?", "installments"),
    ("How much does shipping to Hawaii cost?", "hawaii_price"),
    ("What do you charge to service a bike in your repair shop?", "tune_up"),
    ("Are you open on Saturdays?", "saturdays"),
    ("Are you open on Sundays?", "sundays"),
    ("Until when can I return something I bought?", "return"),
    ("Can I pay in installments?", "installments"),
    ("How long does shipping to Alaska take?", "alaska"),
    ("Is the store open on Saturdays?", "saturdays"),
    ("How much is a tune-up at the repair shop?", "tune_up"),
]


def try_caches(threshold=0.93):
    """Exact cache (same normalized text) versus semantic cache (embedding of the
    most similar question above a threshold)."""
    results = {}
    for kind in ("exact", "semantic"):
        stored, hits, errors = [], 0, 0
        for question, intent in ARRIVALS:
            v = embeddings.encode(["query: " + question], normalize_embeddings=True)[0]
            found = None
            if kind == "exact":
                found = next((i for q, i, _ in stored
                              if q == normalize(question)), None)
            elif stored:
                sims = [float(v @ w) for _, _, w in stored]
                if max(sims) >= threshold:
                    found = stored[sims.index(max(sims))][1]
            if found is None:                  # not there: the model is called
                stored.append((normalize(question), intent, v))
            elif found == intent:
                hits += 1
            else:
                errors += 1
        results[kind] = {"hits": hits, "errors": errors,
                         "model_calls": len(stored)}
        print(f"{kind.capitalize()} cache: {hits} answers reused correctly, "
              f"{errors} wrongly, {len(stored)} model calls out of {len(ARRIVALS)}")
    return results

In [ ]:
caches = try_caches()

## Exercises

1. Explain in your own words why a language model does not remember conversations and how applications solve it.

2. Count the tokens of ten sentences in English and of their Spanish translation. How many tokens per word come out in each language? What does it mean for the cost of an application in Spanish?

3. With the example prices, calculate the monthly cost of the assistant with each memory, assuming 10,000 twelve-message conversations a month. Don't forget the extra calls of the summary and the customer record.

4. Repeat the time measurement on a CPU, in Colab without a GPU. How do the proportions between reading and generating change?

5. Change the sliding window to 4 and 6 turns. How many checks does it pass? How many tokens does it use? From what size does it look like the full history?

6. Add to Martha's conversation, after message 8, a message that changes a fact: "Sorry, my mistake: the bike is an Aero 5". Which memories pick it up? What happens to the customer record, which does not overwrite saved facts? Propose a better rule.

7. Measure the savings of the prefix cache with answers of 10, 40 and 150 tokens. When does it pay off most?

8. Try the semantic cache with thresholds of 0.90, 0.95 and 0.98. Is there one that reuses the paraphrases without confusing Saturdays with Sundays?

9. Write a simple router for the classifier of chapter 2: use Qwen3-0.6B with ten examples and, if its answer is not a valid category, repeat the request with Qwen2.5-1.5B. How many requests reach the large model? How much time is saved?

10. **The assistant's budget.** Write a short report with the estimated cost of handling 10,000 conversations a month with the assistant of chapters 5 and 6 and the memory of your choice, with the example prices, and propose three measures to reduce it.

## References

Models [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct), [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B) and [multilingual-e5-base](https://huggingface.co/intfloat/multilingual-e5-base). *Lost in the middle*: [Liu et al. (2023)](https://arxiv.org/abs/2307.03172). KV cache and vLLM: [Kwon et al. (2023)](https://arxiv.org/abs/2309.06180). The North Cycles store and the conversation are fictional. The prices are examples. Original code and experiments.